# Speech Emotion Recognition - train and compare all models (Kaggle / Colab)

**Before running (Kaggle):**
1. Settings -> Accelerator -> **GPU T4 x2** (or P100). Settings -> **Internet on**.
2. Add Input -> search and add these two datasets:
   - `uwrfkaggler/ravdess-emotional-speech-audio`
   - `ejlok1/cremad`
3. Run all cells. Finished runs are skipped on re-run, so if the session stops, just run again.

**Output:** `reports/` (leaderboard + charts) and `export/` (ONNX model for the backend). Download `ser_outputs.zip` at the end.

In [ ]:
REPO_URL = "https://github.com/Sanjeev2004/Speech-Emotion-Recognization.git"   # change if your repo differs
!test -d repo && (cd repo && git pull -q) || git clone -q $REPO_URL repo
%cd repo/ml
!pip install -q -r requirements.txt

In [ ]:
import glob, os
# Kaggle paths. On Colab, download the datasets and point these to the folders.
def find_dir(marker):
    """Kaggle mounts inputs at different depths over time; locate the folder that holds `marker`."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True))
    return os.path.dirname(hits[-1]) if hits else None

RAVDESS_DIR = find_dir("Actor_01")             # any folder with Actor_xx works
CREMAD_DIR  = find_dir("AudioWAV") and find_dir("AudioWAV") + "/AudioWAV"
print(os.listdir("/kaggle/input"))
assert RAVDESS_DIR and CREMAD_DIR, "Add both datasets via Add Input (see top cell)"
print(len(glob.glob(RAVDESS_DIR + "/**/*.wav", recursive=True)), "RAVDESS wav files")
print(len(glob.glob(CREMAD_DIR + "/**/*.wav", recursive=True)), "CREMA-D wav files")
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
ARGS = f"--ravdess-dir {RAVDESS_DIR} --cremad-dir {CREMAD_DIR} --cache-dir {WORK}/cache --out {WORK}/runs --report-dir {WORK}/reports"

## 1. Check the data split
Speaker-independent: no actor appears in more than one of train / val / test.

In [ ]:
from ser.datasets import build_manifest
df = build_manifest(RAVDESS_DIR, CREMAD_DIR)
display(df.groupby(["dataset", "split"]).size().unstack())
display(df.groupby(["dataset", "emotion"]).size().unstack())

## 2. Classical + spectrogram models (fast, ~1 hour total on GPU)
First run extracts and caches features (one-time, ~10 min).

In [ ]:
!python -m ser.run_all $ARGS --models svm random_forest bilstm cnn2d cnn_lstm crnn_attention

## 3. Self-supervised transformer models (slow, ~30-60 min per run)
Tip: if GPU time is short, run only `--datasets ravdess cremad`, or add `--freeze-backbone` (faster, a bit less accurate).

In [ ]:
!python -m ser.run_all $ARGS --models wav2vec2 hubert wavlm

## 4. Leaderboard

In [ ]:
from IPython.display import Image, Markdown, display
!python -m ser.compare --runs $WORK/runs --out $WORK/reports
display(Markdown(open(f"{WORK}/reports/leaderboard.md").read()))
display(Image(f"{WORK}/reports/uar_by_model.png"))
if os.path.exists(f"{WORK}/reports/cross_dataset.png"):
    display(Image(f"{WORK}/reports/cross_dataset.png"))

## 5. Export the best small model for deployment
The free Render server has 512 MB RAM, so only spectrogram models (a few MB) are deployed.
SSL models stay in the comparison.

In [ ]:
import pandas as pd
from ser.models import REGISTRY
lb = pd.read_csv(f"{WORK}/reports/leaderboard.csv")
ok = lb[(lb.train_dataset == "combined") & (lb.test_dataset == "combined")
        & lb.model.map(lambda m: REGISTRY[m].deployable)]
best = ok.sort_values("uar", ascending=False).iloc[0]
print("Deploying:", best.model, "UAR", round(best.uar, 3))
!python -m ser.export_onnx --run $WORK/runs/{best.model}__combined --out $WORK/export --leaderboard $WORK/reports/leaderboard.json

In [ ]:
!cd $WORK && zip -qr ser_outputs.zip reports export runs -x "runs/*/best.pt" "runs/*/model.joblib"
print("Download", f"{WORK}/ser_outputs.zip", "-> unzip: export/* into backend/models/")